# Eval - fixed autonomous evaluation

This notebook loads a dataset provider plus the immutable training/metric code, tunes `Rules.py` parameters with SciPy, and records permutation-invariant hierarchy recovery. Set `AUTORD_DATASET_NOTEBOOK` before launch to use another provider implementing the contract documented in `Dataset.ipynb`. **The coding agent must not edit this notebook during Auto-R&D.**

In [ ]:
from pathlib import Path
import importlib.util
import json
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import differential_evolution
import nbformat


def locate_notebook_dir():
    candidates = (Path.cwd(), Path.cwd() / 'nb')
    for candidate in candidates:
        if (candidate / 'Prepare.ipynb').is_file():
            return candidate.resolve()
    raise FileNotFoundError('run from the repository root or nb directory')


def execute_notebook(path, namespace=None):
    namespace = {} if namespace is None else namespace
    notebook = nbformat.read(path, as_version=4)
    for index, cell in enumerate(notebook.cells):
        tags = set(cell.metadata.get('tags', ()))
        if cell.cell_type == 'code' and 'skip-on-provider-import' not in tags:
            exec(compile(cell.source, f'{path}:cell-{index}', 'exec'), namespace)
    return namespace


NB_DIR = locate_notebook_dir()
provider_setting = os.environ.get('AUTORD_DATASET_NOTEBOOK', 'Dataset.ipynb')
DATASET_NOTEBOOK = Path(provider_setting)
if not DATASET_NOTEBOOK.is_absolute():
    DATASET_NOTEBOOK = NB_DIR / DATASET_NOTEBOOK
DATASET_NOTEBOOK = DATASET_NOTEBOOK.resolve()

benchmark_ns = execute_notebook(DATASET_NOTEBOOK)
required_provider_names = {
    'DATASET_PROVIDER_NAME', 'DEV_CASES', 'PROMOTION_CASES',
    'DEV_SEEDS', 'PROMOTION_SEEDS', 'case_name', 'load_dataset',
}
missing_provider_names = sorted(required_provider_names.difference(benchmark_ns))
if missing_provider_names:
    raise RuntimeError(
        f'dataset provider {DATASET_NOTEBOOK} is missing: {missing_provider_names}'
    )
execute_notebook(NB_DIR / 'Prepare.ipynb', benchmark_ns)
globals().update(benchmark_ns)


def load_rules():
    rule_path = NB_DIR / 'Rules.py'
    spec = importlib.util.spec_from_file_location('Rules', rule_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


print(f'dataset provider: {DATASET_PROVIDER_NAME} ({DATASET_NOTEBOOK})')

In [ ]:
def materialize_panel(cases, seeds):
    materialized = []
    for case_index, case in enumerate(cases):
        for seed in seeds:
            dataset = prepare_dataset(load_dataset(case, seed))
            materialized.append((case_index, case_name(case), seed, dataset))
    return materialized


DEV_PANEL = materialize_panel(DEV_CASES, DEV_SEEDS)
PROMOTION_PANEL = materialize_panel(PROMOTION_CASES, PROMOTION_SEEDS)


def cfg_from_vector(rule, names, values):
    return rule.RuleConfig(**{name: float(value) for name, value in zip(names, values)})


def panel(rule, materialized_panel, cfg):
    rows = []
    for case_index, dataset_name, seed, dataset in materialized_panel:
        result = evaluate_dataset(rule, dataset, training_seed=seed, cfg=cfg)
        result.update(dataset=dataset_name, case=case_index, seed=seed)
        rows.append(result)
    return rows


def fitness_from_rows(rule, rows):
    values = np.array([row['overall'] for row in rows], float)
    if not all(row['stable'] for row in rows):
        return -1e6
    return float(
        values.mean() - 0.35 * values.std() + 0.15 * values.min()
        - 0.002 * rule.complexity_score()
    )


def optimize_params(maxiter=10, popsize=6):
    rule = load_rules()
    names = list(rule.PARAM_BOUNDS)
    bounds = [rule.PARAM_BOUNDS[name] for name in names]

    def objective(values):
        cfg = cfg_from_vector(rule, names, values)
        return -fitness_from_rows(rule, panel(rule, DEV_PANEL, cfg))

    result = differential_evolution(
        objective, bounds, seed=123, maxiter=maxiter, popsize=popsize,
        polish=True, workers=1,
    )
    return rule, names, result


# Short smoke run. Promotion-quality runs require an operator-designated fixed budget.
rule, names, optimization = optimize_params(maxiter=2, popsize=3)
best_cfg = cfg_from_vector(rule, names, optimization.x)
print('best', dict(zip(names, optimization.x)), 'dev fitness', -optimization.fun)

In [ ]:
promotion_rows = panel(rule, PROMOTION_PANEL, best_cfg)
promotion_fitness = fitness_from_rows(rule, promotion_rows)
df = pd.DataFrame([
    {key: value for key, value in row.items() if key != 'cross'}
    for row in promotion_rows
])
display(df)
print('promotion fitness:', promotion_fitness)
print(df[['overall', 'hierarchy', 'specificity', 'topology']].agg(['mean', 'std', 'min']))

In [ ]:
# Visualize cross-level correspondence scores for one representative run.
representative = promotion_rows[0]
plt.figure(figsize=(5, 4))
plt.imshow(representative['cross'], aspect='auto')
plt.xlabel('Dataset hierarchy level')
plt.ylabel('Neural layer')
plt.title('Permutation-invariant cross-level correspondence')
plt.colorbar(label='assignment score')
plt.show()

In [ ]:
# Append a compact trial result. Rule snapshots are managed separately under out/rules/.
path = NB_DIR / 'experiments.csv'
record = {
    'timestamp': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'dataset_provider': DATASET_PROVIDER_NAME,
    'fitness': promotion_fitness,
    'mean_overall': df.overall.mean(),
    'std_overall': df.overall.std(),
    'min_overall': df.overall.min(),
    'mean_hierarchy': df.hierarchy.mean(),
    'mean_specificity': df.specificity.mean(),
    'mean_topology': df.topology.mean(),
    'params': json.dumps(best_cfg.__dict__, sort_keys=True),
}
pd.DataFrame([record]).to_csv(path, mode='a', header=not path.exists(), index=False)
record